In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import time
import pickle
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import norm

from b_Closed_form import *
from c_MC import *
from d_FDM import *
from e_0_generate import *
from e_1_run_cvae import * 
from e_2_CVAE import *

# global var
S0 = 1.0
K = 1.0
r = 0.03
sigma = np.sqrt(0.05)
T = 1.5
BS_eta = (S0, K, r, sigma, T)
# S0, K, r, kappa, theta, xi, rho, Y0, T = Hes_eta
Hes_eta = (S0, K, r, 2, 0.05, 0.5, -0.7, 0.05, T)

B = 0.8 # down-and-out must B < S0 and B < K
model_type = 'bs' # hes or bs
barr_type = 'van' # van or barr
opt_type = 'call' # call or put
chunk_dir = f"/mnt/d/bs_chunks_correction2/" if model_type == 'bs' else f"/mnt/d/hes_chunks_correction2/"


if not((B < S0) & (B < K)):
    raise ValueError("down-and-out : B should be smaller than S0 and K")

if not(opt_type == 'call' or  opt_type == 'put'):
    raise ValueError("option_type must be 'call' or 'put'")

if not(barr_type == 'van' or  barr_type == 'barr'):
    raise ValueError("barr_type must be 'van' or 'barr'")

if not(model_type == 'hes' or  model_type == 'bs'):
    raise ValueError("model_type must be 'hes' or 'bs'")

# cvae training settings
if model_type == 'hes':
    if barr_type == 'barr':
        if opt_type == 'call':
            bench_price = 0.115733
        else: # put
            bench_price = 0.005170
    else: # van
        if opt_type == 'call':
            bench_price = 0.124491
        else: # put
            bench_price = 0.080488

elif model_type == 'bs':
    if barr_type == 'barr':
        if opt_type == 'call':
            bench_price = 0.123493
        else: # put
            bench_price = 0.009535
    else: # van
        if opt_type == 'call':
            bench_price = 0.129944
        else: # put
            bench_price = 0.085942

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# 1. closed-form pricing result
closed_bs_van = BS_vanilla(BS_eta, opt_type)
closed_bs_barr = BS_barrier(BS_eta, B, opt_type)

In [ ]:
# 2. MC
mc_bs_van_gpu = MC_BS_vanilla_gpu(BS_eta, n_paths=100000, type=opt_type)
mc_hes_van_gpu = MC_heston_vanilla_gpu(Hes_eta, n_paths=100000, type=opt_type)
# BS_van
# CPU vanilla(1k, 10k, 100k) = (0.042, 0.453, 4.783)
# GPU vanilla(1k, 10k, 100k) = (0.143, 0.156, 0.163)

# Hes_van
# CPU vanilla(1k, 10k, 100k) = (0.112, 1.160, 11.992)
# GPU vanilla(1k, 10k, 100k) = (0.438, 0.485, 0.54)

#start = time.time()
mc_bs_barr_gpu = MC_BS_barrier_gpu(BS_eta, B, n_paths=100000, type=opt_type)
mc_hes_barr_gpu = MC_heston_barrier_gpu(Hes_eta, B, n_paths=100000, type=opt_type)
#print(f"{time.time() - start:.3f}s")
# BS_barr
# CPU vanilla(1k, 10k, 100k) = ( , , )
# GPU vanilla(1k, 10k, 100k) = ( 0.150, 0.150, 0.155)

# Hes_barr
# CPU vanilla(1k, 10k, 100k) = ( , , )
# GPU vanilla(1k, 10k, 100k) = ( 0.45, 0.47, 0.53)

In [ ]:
# 3. FDM
# vanilla
#ftcs_bs_van = FTCS_BS_vanilla(BS_eta, opt_type)
cn_bs_van = CN_BS_vanilla(BS_eta, opt_type)
cs_hes_van = CS_ADI_heston_vanilla(Hes_eta, opt_type)
#cs_hes_van = CS_ADI_heston_vanilla(Hes_eta, opt_type, dS=0.01, dv=0.0001, dt=0.01) # put인경우

# barrier
#ftcs_bs_barr = FTCS_BS_barrier(BS_eta, opt_type, B)
cn_bs_barr = CN_BS_barrier(BS_eta, opt_type, B)
cs_hes_barr = CS_ADI_heston_barrier(Hes_eta, opt_type, B)
#cs_hes_barr = CS_ADI_heston_barrier(Hes_eta, opt_type, B, dS=0.01, dv=0.0001, dt=0.01) # put인경우

In [ ]:
# 4. Gerate dataset for CVAE
# step1-1 BS
BS_paras = generate_BS_params(n_sets=100*(2**16), seed=1234)
bs_eta_dim = BS_paras.shape[1]

with h5py.File(BS_ETA_PATH, "w") as f:
    f.create_dataset("etas", data=BS_paras,
                    maxshape=(None, bs_eta_dim), 
                    chunks=(10240, bs_eta_dim), 
                    compression="gzip"
                    )

In [ ]:
# step2
# VANILLA X_T만 / Barrier X_T, M_T
eta_chunk_size = 2**16 
generate_dataset(BS_ETA_PATH, "/mnt/d/bs_chunks_correction2/", 
                 model_type='bs', S0=S0, B=B, 
                 chunk_size=eta_chunk_size
                 ) # 1 ROUND : 3.8시간

round:9
[20000/655360] fail: 5132
[40000/655360] fail: 10453
[60000/655360] fail: 15751
  청크 저장: /mnt/d/bs_chunks_correction/bs_chunk_090.h5 (67,108,864행)
[80000/655360] fail: 21083
[100000/655360] fail: 26281
[120000/655360] fail: 31530
  청크 저장: /mnt/d/bs_chunks_correction/bs_chunk_091.h5 (67,108,864행)
[140000/655360] fail: 36964
[160000/655360] fail: 42354
[180000/655360] fail: 47752
  청크 저장: /mnt/d/bs_chunks_correction/bs_chunk_092.h5 (67,108,864행)
[200000/655360] fail: 53049
[220000/655360] fail: 58405
[240000/655360] fail: 63701
[260000/655360] fail: 69086
  청크 저장: /mnt/d/bs_chunks_correction/bs_chunk_093.h5 (67,108,864행)
[280000/655360] fail: 74402
[300000/655360] fail: 79864
[320000/655360] fail: 85289
  청크 저장: /mnt/d/bs_chunks_correction/bs_chunk_094.h5 (67,108,864행)
[340000/655360] fail: 90639
[360000/655360] fail: 95833
[380000/655360] fail: 101175
  청크 저장: /mnt/d/bs_chunks_correction/bs_chunk_095.h5 (67,108,864행)
[400000/655360] fail: 106633
[420000/655360] fail: 112008
[440

In [ ]:
# step1-2 Hes
Hes_paras = generate_hes_valid_params(n_sets=100*(2**16), seed=1234) # 4.1초
hes_eta_dim = Hes_paras.shape[1]

with h5py.File(HES_ETA_PATH, "w") as f:
    f.create_dataset("etas", data=Hes_paras,
                    maxshape=(None, hes_eta_dim), 
                    chunks=(10240, hes_eta_dim), 
                    compression="gzip"
                    )

In [ ]:
# step2
eta_chunk_size = 2**16
generate_dataset(HES_ETA_PATH, "/mnt/d/heston_chunks_correction2/", 
                 model_type='hes', S0=S0, B=B, 
                 chunk_size=eta_chunk_size
                 ) # 1ROUND : 5시간

In [6]:
# compute for normalization
#bs_stats = compute_xm_stats(model_type="bs") 
# bs : x_mean=-0.1045446063, std=0.6455563393 / m_mean=-0.4579059199, std=0.5553496410
hes_stats = compute_xm_stats(model_type="hes")

[001/100] done: heston_chunk_000.h5 rows=67,108,864
[002/100] done: heston_chunk_001.h5 rows=67,108,864
[003/100] done: heston_chunk_002.h5 rows=67,108,864
[004/100] done: heston_chunk_003.h5 rows=67,108,864
[005/100] done: heston_chunk_004.h5 rows=67,108,864
[006/100] done: heston_chunk_005.h5 rows=67,108,864
[007/100] done: heston_chunk_006.h5 rows=67,108,864
[008/100] done: heston_chunk_007.h5 rows=67,108,864
[009/100] done: heston_chunk_008.h5 rows=67,108,864
[010/100] done: heston_chunk_009.h5 rows=67,108,864
[011/100] done: heston_chunk_010.h5 rows=67,108,864
[012/100] done: heston_chunk_011.h5 rows=67,108,864
[013/100] done: heston_chunk_012.h5 rows=67,108,864
[014/100] done: heston_chunk_013.h5 rows=67,108,864
[015/100] done: heston_chunk_014.h5 rows=67,108,864
[016/100] done: heston_chunk_015.h5 rows=67,108,864
[017/100] done: heston_chunk_016.h5 rows=67,108,864
[018/100] done: heston_chunk_017.h5 rows=67,108,864
[019/100] done: heston_chunk_018.h5 rows=67,108,864
[020/100] do

================== result ===================

In [ ]:
# Total result
#vanilla
print(closed_bs_van, mc_bs_van_gpu, cn_bs_van)
print(f"{(mc_bs_van_gpu - closed_bs_van) / closed_bs_van * 100}%")
print(f"{(cn_bs_van - closed_bs_van) / closed_bs_van * 100}%\n")

print(mc_hes_van_gpu, cs_hes_van)
print(f"{(cs_hes_van - mc_hes_van_gpu) / mc_hes_van_gpu * 100}%\n\n")

#barrier
print(closed_bs_barr, mc_bs_barr_gpu, cn_bs_barr)
print(f"{(mc_bs_barr_gpu - closed_bs_barr) / closed_bs_barr * 100}%")
print(f"{(cn_bs_barr - closed_bs_barr) / closed_bs_barr * 100}%\n")

print(mc_hes_barr_gpu, cs_hes_barr)
print(f"{(cs_hes_barr - mc_hes_barr_gpu) / mc_hes_barr_gpu * 100}%")


In [ ]:
'''
# MC
n_list    = [1000, 10000, 100000]
n_repeats = 50
results   = {n: [] for n in n_list}

for n in n_list:
    for _ in range(n_repeats):
        price = MC_heston_barrier_gpu(Hes_eta, B, n_paths=n, type=opt_type) # MC_heston_barrier_gpu
        results[n].append(price)

#MC_BS_vanilla_gpu, MC_heston_vanilla_gpu, MC_BS_barrier_gpu, MC_heston_barrier_gpu
#BS_eta, Hes_eta

# pickle은 binary로.
with open(f'mc_{model_type}_{barr_type}_{opt_type}_results.pkl', 'wb') as f:
    pickle.dump(results, f)
'''

"\nn_list    = [1000, 10000, 100000]\nn_repeats = 50\nresults   = {n: [] for n in n_list}\n\nfor n in n_list:\n    for _ in range(n_repeats):\n        price = MC_heston_barrier_gpu(Hes_eta, B, n_paths=n, type=opt_type) # MC_heston_barrier_gpu\n        results[n].append(price)\n\n#MC_BS_vanilla_gpu, MC_heston_vanilla_gpu, MC_BS_barrier_gpu, MC_heston_barrier_gpu\n#BS_eta, Hes_eta\n\n# pickle은 binary로.\nwith open('mc_hes_barr_put_results.pkl', 'wb') as f:\n    pickle.dump(results, f)\n"

In [ ]:
'''
# FDM
configs = [ 
# vanilla
    # call
        dict(dS=0.01, dv=0.01, dt=0.01),  # = 0.12997582
        dict(dS=0.01, dv=0.001, dt=0.01), # = 0.12512163, 22s ***
        dict(dS=0.01, dv=0.0001, dt=0.01), # = 0.12449123, 3m 44s *****
    dict(dS=0.005, dv=0.005, dt=0.005), # = 0.127356
    dict(dS=0.002, dv=0.002, dt=0.005), # = 0.125731, dt=0.01에서 줄여도 큰 차이 없음
    dict(dS=0.002, dv=0.002, dt=0.01), # = 0.125723
    dict(dS=0.002, dv=0.001, dt=0.01), # = 0.125134
    dict(dS=0.001, dv=0.002, dt=0.01), # = 0.125692, 메모리 10% 사용, dv 줄이는게 영향이 더 큼
    dict(dS=0.001, dv=0.001, dt=0.01), # = 0.125106
    dict(dS=0.001, dv=0.0001, dt=0.01), # = 0.124478, 28m, 38% 사용
    dict(dS=0.01, dv=0.00001, dt=0.01), # = 0.124395, 40m 41s
    dict(dS=0.0005, dv=0.0005, dt=0.01), # = 0.124748, 1step 5초
    dict(dS=0.0001, dv=0.0005, dt=0.01), # memory error
    dict(dS=0.0001, dv=0.0001, dt=0.01), # memory error
    
    #put
        dict(dS=0.01, dv=0.01, dt=0.01), # = 0.08597330
        dict(dS=0.01, dv=0.001, dt=0.01), # = 0.08111911
        dict(dS=0.01, dv=0.0001, dt=0.01), # = 0.08048871 *****

        
        
 # barr
    # call
        dict(dS=0.01, dv=0.01, dt=0.01), # = 0.12014952
        dict(dS=0.01, dv=0.001, dt=0.01), # = 0.11625772, 20.3s
        dict(dS=0.01, dv=0.0001, dt=0.01), # = 0.11573391, 3m 37s ***
    dict(dS=0.01, dv=0.00001, dt=0.01), # = 0.115654, 44m 26s
    dict(dS=0.001, dv=0.0001, dt=0.01), # = 0.115722, 21m 20s
    
    # put
        dict(dS=0.01, dv=0.01, dt=0.01), # = 0.00539354, 3.6s, 
        dict(dS=0.01, dv=0.001, dt=0.01), # = 0.00521346, 20.6s, 
        dict(dS=0.01, dv=0.0001, dt=0.01), # = 0.00517056, 3m 22s,
    dict(dS=0.001, dv=0.01, dt=0.01), # = 0.005410,  
    dict(dS=0.0001, dv=0.01, dt=0.01), # = 0.005410
    dict(dS=0.001, dv=0.001, dt=0.01), # =0.005231
]
'''
'''
configs = [ 
    dict(dS=0.01, dv=0.01, dt=0.01),
    dict(dS=0.01, dv=0.001, dt=0.01),
    dict(dS=0.01, dv=0.0001, dt=0.01),
]

for cfg in configs:
    p = CS_ADI_heston_vanilla(Hes_eta, type=opt_type, **cfg)
    print(cfg, f"→ {p:.8f}")
'''

In [ ]:
# 공용 그래프
n_list    = [1000, 10000, 100000]
method = 'cvae' # 'cvae', 'mc'
with open(f'{method}_{model_type}_{barr_type}_{opt_type}_results.pkl', 'rb') as f:
    results = pickle.load(f)
    
means  = [np.mean(results[n]) for n in n_list]
stds   = [np.std(results[n])  for n in n_list]
ci     = [1.96 * s for s in stds]

plt.figure(figsize=(7, 4))
plt.axhline(bench_price, color='gray', linewidth=1.5, label='FDM')
plt.errorbar(range(len(n_list)), means, yerr=ci,
             fmt='o-', color='steelblue', capsize=5, label=f'{method} (95% CI)')
plt.xticks(range(len(n_list)), ['1K', '10K', '100K'])
plt.xlabel('Number of simulations')
plt.ylabel('Option price')
plt.title('convergence')
plt.legend()
plt.tight_layout()